# QC Transfer Benchmark

Runs the hybrid-to-paired transfer benchmark with live progress output.

- Benchmark progress is printed split-by-split with elapsed time and ETA.
- Neural transfer prints per-epoch losses and ETA so you can see how long it will take.
- Use the run profiles below instead of the old smoke toggle.

In [6]:
from pathlib import Path
import sys

from IPython.display import display

repo_root = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from qc_framework import TransferBenchmarkConfig, QCTransferBenchmark

## Configuration

- `meaningful_baseline` is the safest first real run. It excludes the research neural family and focuses on the transfer baselines most likely to beat dummy.
- `neural_probe` keeps the baseline ladder and adds the neural transfer family with moderate epochs and live epoch logs.
- `full_research` is the expensive run once the baseline path is already stable.

In [7]:
PARQUET_PATH = repo_root / 'data/raw/33000_ROWS.parquet'
OUTPUT_DIR = repo_root / 'analysis/runs/transfer_notebook/outputs'

RUN_PROFILE = 'meaningful_baseline'

PROFILES = {
    'meaningful_baseline': {
        'model_families': ('dummy', 'paired_only', 'hybrid_only', 'hybrid_calibrated', 'hybrid_stack', 'hybrid_plus_paired'),
        'paired_models': ('lightgbm',),
        'model_selection_splits': 3,
        'paired_weight_grid': (10.0, 25.0),
        'lightgbm_estimators': 120,
        'neural_pretrain_epochs': 0,
        'neural_finetune_epochs': 0,
        'neural_batch_size': 128,
    },
    'neural_probe': {
        'model_families': ('dummy', 'paired_only', 'hybrid_only', 'hybrid_calibrated', 'hybrid_stack', 'hybrid_plus_paired', 'neural_transfer'),
        'paired_models': ('lightgbm',),
        'model_selection_splits': 2,
        'paired_weight_grid': (10.0, 25.0),
        'lightgbm_estimators': 120,
        'neural_pretrain_epochs': 8,
        'neural_finetune_epochs': 5,
        'neural_batch_size': 128,
    },
    'full_research': {
        'model_families': ('dummy', 'paired_only', 'hybrid_only', 'hybrid_calibrated', 'hybrid_stack', 'hybrid_plus_paired', 'neural_transfer'),
        'paired_models': ('lightgbm', 'catboost'),
        'model_selection_splits': 5,
        'paired_weight_grid': (10.0, 25.0, 50.0),
        'lightgbm_estimators': 250,
        'neural_pretrain_epochs': 35,
        'neural_finetune_epochs': 20,
        'neural_batch_size': 128,
    },
}

profile = PROFILES[RUN_PROFILE]

config = TransferBenchmarkConfig(
    parquet_path=PARQUET_PATH,
    paired_final_holdout_rows=100,
    model_selection_splits=profile['model_selection_splits'],
    model_families=profile['model_families'],
    paired_models=profile['paired_models'],
    paired_weight_grid=profile['paired_weight_grid'],
    lightgbm_estimators=profile['lightgbm_estimators'],
    neural_pretrain_epochs=profile['neural_pretrain_epochs'],
    neural_finetune_epochs=profile['neural_finetune_epochs'],
    neural_batch_size=profile['neural_batch_size'],
    verbose=True,
    neural_verbose=True,
)

print('RUN_PROFILE =', RUN_PROFILE)
config

RUN_PROFILE = meaningful_baseline


TransferBenchmarkConfig(parquet_path=PosixPath('/Users/paulruiz/Documents/Predicting_Good_Units/data/raw/33000_ROWS.parquet'), primary_targets=('fpos', 'fmiss'), secondary_targets=('accuracy',), source_fmiss_target='fmiss_extended', final_eval_fmiss_target='fmiss', paired_final_holdout_rows=100, allow_unlabeled_paired=True, feature_views=('norm_swap', 'shape_only', 'recording_relative'), model_families=('dummy', 'paired_only', 'hybrid_only', 'hybrid_calibrated', 'hybrid_stack', 'hybrid_plus_paired'), feature_view_sets=(('shape_only', ('shape_only',)), ('norm_swap', ('norm_swap',)), ('norm_swap+recording_relative', ('norm_swap', 'recording_relative'))), model_selection_splits=3, model_selection_test_size=0.4, random_state=42, lightgbm_estimators=120, lightgbm_learning_rate=0.05, lightgbm_num_leaves=31, paired_models=('lightgbm',), paired_weight_grid=(10.0, 25.0), neural_pretrain_epochs=0, neural_finetune_epochs=0, neural_batch_size=128, neural_lr=0.001, neural_domain_loss_weight=0.15, n

## Run

Execute the next cell and watch the output. In notebook mode you will see:

- current split / target / family
- elapsed time and benchmark ETA
- neural pretrain and finetune epoch losses with ETA

In [8]:
runner = QCTransferBenchmark(config)
artifacts = runner.run()

Transfer benchmark start | paired matched=207 | hybrid=24211 | paired=8982 | total tasks=72
[1/72] model_selection split=1 target=fpos family=dummy ...
[1/72] model_selection split=1 target=fpos family=dummy done in 0.0s | elapsed=0.0m | eta=0.0m
[2/72] model_selection split=1 target=fpos family=paired_only ...
[2/72] model_selection split=1 target=fpos family=paired_only done in 0.2s | elapsed=0.0m | eta=0.1m
[3/72] model_selection split=1 target=fpos family=hybrid_only ...
[3/72] model_selection split=1 target=fpos family=hybrid_only done in 2.8s | elapsed=0.1m | eta=1.2m
[4/72] model_selection split=1 target=fpos family=hybrid_calibrated ...
[4/72] model_selection split=1 target=fpos family=hybrid_calibrated done in 1.9s | elapsed=0.1m | eta=1.4m
[5/72] model_selection split=1 target=fpos family=hybrid_stack ...
[5/72] model_selection split=1 target=fpos family=hybrid_stack done in 1.9s | elapsed=0.1m | eta=1.5m
[6/72] model_selection split=1 target=fpos family=hybrid_plus_paired ..

## Review Results

In [11]:
display(artifacts['config'])
display(artifacts['feature_leakage_checks'])
display(artifacts['split_report'])
display(artifacts['model_selection_summary'].sort_values(['primary_avg_mae', 'candidate_id', 'target']).head(30))
display(artifacts['final_holdout_summary'].sort_values(['primary_avg_mae', 'candidate_id', 'target']).head(30))
display(artifacts['winner_summary'])

,parquet_path,primary_targets,secondary_targets,source_fmiss_target,final_eval_fmiss_target,paired_final_holdout_rows,allow_unlabeled_paired,feature_views,model_families,feature_view_sets,...,paired_models,paired_weight_grid,neural_pretrain_epochs,neural_finetune_epochs,neural_batch_size,neural_lr,neural_domain_loss_weight,neural_target_loss_weight,verbose,neural_verbose
0,/Users/paulruiz/Documents/Predicting_Good_Unit...,"(fpos, fmiss)","(accuracy,)",fmiss_extended,fmiss,100,True,"(norm_swap, shape_only, recording_relative)","(dummy, paired_only, hybrid_only, hybrid_calib...","((shape_only, (shape_only,)), (norm_swap, (nor...",...,"(lightgbm,)","(10.0, 25.0)",0,0,128,0.001,0.15,2.0,True,True


,feature_view,n_features,raw_normalized_overlap,passes_norm_swap_check
0,shape_only,70,0,1
1,norm_swap,215,0,1
2,norm_swap+recording_relative,220,0,1


,paired_matched_rows,target_final_holdout_rows,actual_final_holdout_rows,final_train_rows,final_holdout_recordings,final_train_recordings,recording_overlap
0,207,100,100,107,15,14,0


,candidate_id,model_family,model_name,feature_view,target,mae,rmse,r2,bias,calibration_slope,calibration_intercept,primary_avg_mae,dummy_mae,best_hybrid_only_mae,beats_dummy,beats_best_hybrid_only,beats_dummy_all_primary_targets,beats_hybrid_only_all_primary_targets,advances
15,hybrid_plus_paired|lightgbm_w10|norm_swap+reco...,hybrid_plus_paired,lightgbm_w10,norm_swap+recording_relative,accuracy,0.200027,0.252264,0.155938,-0.066227,1.158012,-0.023613,0.199672,NaN,NaN,False,False,True,True,True
16,hybrid_plus_paired|lightgbm_w10|norm_swap+reco...,hybrid_plus_paired,lightgbm_w10,norm_swap+recording_relative,fmiss,0.235498,0.303547,-0.472431,0.048741,0.622770,0.113429,0.199672,0.246339,0.287686,True,True,True,True,True
17,hybrid_plus_paired|lightgbm_w10|norm_swap+reco...,hybrid_plus_paired,lightgbm_w10,norm_swap+recording_relative,fpos,0.163846,0.208789,0.099558,0.062900,0.830074,-0.016112,0.199672,0.192789,0.225079,True,True,True,True,True
18,hybrid_plus_paired|lightgbm_w25|norm_swap+reco...,hybrid_plus_paired,lightgbm_w25,norm_swap+recording_relative,accuracy,0.191873,0.247898,0.214340,-0.062481,1.006262,0.056649,0.201357,NaN,NaN,False,False,True,True,True
19,hybrid_plus_paired|lightgbm_w25|norm_swap+reco...,hybrid_plus_paired,lightgbm_w25,norm_swap+recording_relative,fmiss,0.229347,0.302626,-0.407565,0.052598,0.505222,0.134548,0.201357,0.246339,0.287686,True,True,True,True,True
20,hybrid_plus_paired|lightgbm_w25|norm_swap+reco...,hybrid_plus_paired,lightgbm_w25,norm_swap+recording_relative,fpos,0.173367,0.224904,-0.051270,0.083583,0.706119,0.002839,0.201357,0.192789,0.225079,True,True,True,True,True
27,paired_only|lightgbm|shape_only,paired_only,lightgbm,shape_only,accuracy,0.235330,0.276005,0.094490,-0.015632,0.780471,0.145059,0.201748,NaN,NaN,False,False,False,True,False
28,paired_only|lightgbm|shape_only,paired_only,lightgbm,shape_only,fmiss,0.247352,0.306630,-0.273147,-0.031460,0.531797,0.180700,0.201748,0.246339,0.287686,False,True,False,True,False
29,paired_only|lightgbm|shape_only,paired_only,lightgbm,shape_only,fpos,0.156145,0.199761,0.147591,0.055869,0.820098,-0.002322,0.201748,0.192789,0.225079,True,True,False,True,False
3,hybrid_calibrated|lightgbm+isotonic|norm_swap+...,hybrid_calibrated,lightgbm+isotonic,norm_swap+recording_relative,accuracy,0.253629,0.307284,-0.237241,-0.031241,0.874946,0.027582,0.209251,NaN,NaN,False,False,False,True,False


,candidate_id,model_family,model_name,feature_view,target,mae,rmse,r2,bias,calibration_slope,calibration_intercept,primary_avg_mae,advances,beats_dummy_all_primary_targets,beats_hybrid_only_all_primary_targets,beats_best_paired_only
3,hybrid_calibrated|lightgbm+isotonic|norm_swap+...,hybrid_calibrated,lightgbm+isotonic,norm_swap+recording_relative,accuracy,0.208233,0.259804,0.284321,-0.006645,0.839624,0.105661,0.174846,False,False,True,True
4,hybrid_calibrated|lightgbm+isotonic|norm_swap+...,hybrid_calibrated,lightgbm+isotonic,norm_swap+recording_relative,fmiss,0.211667,0.260874,0.188873,0.008264,0.883623,0.024995,0.174846,False,False,True,True
5,hybrid_calibrated|lightgbm+isotonic|norm_swap+...,hybrid_calibrated,lightgbm+isotonic,norm_swap+recording_relative,fpos,0.138025,0.203269,0.319602,-0.017403,1.016093,0.014128,0.174846,False,False,True,True
9,hybrid_plus_paired|lightgbm_w10|norm_swap+reco...,hybrid_plus_paired,lightgbm_w10,norm_swap+recording_relative,accuracy,0.217326,0.273830,0.204961,-0.107140,0.829198,0.195428,0.177280,True,True,True,True
10,hybrid_plus_paired|lightgbm_w10|norm_swap+reco...,hybrid_plus_paired,lightgbm_w10,norm_swap+recording_relative,fmiss,0.210174,0.269168,0.136475,0.070209,0.693366,0.036417,0.177280,True,True,True,True
11,hybrid_plus_paired|lightgbm_w10|norm_swap+reco...,hybrid_plus_paired,lightgbm_w10,norm_swap+recording_relative,fpos,0.144386,0.187149,0.423235,0.067253,1.054584,-0.082982,0.177280,True,True,True,True
12,hybrid_stack|lightgbm_residual|norm_swap+recor...,hybrid_stack,lightgbm_residual,norm_swap+recording_relative,accuracy,0.202033,0.273948,0.204276,-0.088616,0.691061,0.254031,0.179959,False,False,True,True
13,hybrid_stack|lightgbm_residual|norm_swap+recor...,hybrid_stack,lightgbm_residual,norm_swap+recording_relative,fmiss,0.208431,0.282228,0.050645,0.080918,0.584576,0.067987,0.179959,False,False,True,True
14,hybrid_stack|lightgbm_residual|norm_swap+recor...,hybrid_stack,lightgbm_residual,norm_swap+recording_relative,fpos,0.151486,0.199351,0.345576,0.024168,0.753234,0.036309,0.179959,False,False,True,True
15,paired_only|lightgbm|shape_only,paired_only,lightgbm,shape_only,accuracy,0.211453,0.250770,0.333230,-0.025142,1.096175,-0.032457,0.209102,False,False,True,False


,candidate_id,model_family,model_name,feature_view,primary_avg_mae,beats_best_paired_only,advances,eligible_default,best_paired_only_primary_avg_mae
1,hybrid_plus_paired|lightgbm_w10|norm_swap+reco...,hybrid_plus_paired,lightgbm_w10,norm_swap+recording_relative,0.17728,True,True,True,0.209102


## Save Artifacts

In [12]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, obj in artifacts.items():
    if hasattr(obj, 'to_csv'):
        obj.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)

print('Saved to:', OUTPUT_DIR)

Saved to: /Users/paulruiz/Documents/Predicting_Good_Units/analysis/runs/transfer_notebook/outputs
